# CatBoost categorical features issue with ForecasterRecursive

CatBoost has its own internal categorical encoding (target statistics / ordered target encoding). It expects **raw category values** (strings or integers), not pre-encoded floats.

The problem: ForecasterRecursive passes `X_train` as a **numpy float64 array** to `estimator.fit()`. The `OrdinalEncoder` converts categories to float (0.0, 1.0, 2.0...). CatBoost rejects `cat_features` when `X` is a float numpy array.

In [2]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor

## 1. CatBoost with float numpy array — FAILS

In [3]:
# This is what ForecasterRecursive does: X_train is always float64 numpy
X_float = np.array([
    [1.0, 0.0],
    [2.0, 1.0],
    [3.0, 0.0],
    [4.0, 1.0],
    [5.0, 0.0],
], dtype=np.float64)
y = np.array([10.0, 20.0, 15.0, 25.0, 12.0])

model = CatBoostRegressor(verbose=0, iterations=10)
try:
    model.fit(X_float, y, cat_features=[1])
    print('SUCCESS')
except Exception as e:
    print(f'ERROR: {e}')

ERROR: 'data' is numpy array of floating point numerical type, it means no categorical features, but 'cat_features' parameter specifies nonzero number of categorical features


## 2. CatBoost with int numpy array — WORKS

In [4]:
X_int = X_float.copy()
X_int[:, 1] = X_int[:, 1].astype(int)
print(f'dtype: {X_int.dtype}')
# Still float64 because the first column has floats
# CatBoost will still reject it

dtype: float64


In [5]:
model = CatBoostRegressor(verbose=0, iterations=10)
try:
    model.fit(X_int, y, cat_features=[1])
    print('SUCCESS')
except Exception as e:
    print(f'ERROR: {e}')

ERROR: 'data' is numpy array of floating point numerical type, it means no categorical features, but 'cat_features' parameter specifies nonzero number of categorical features


## 3. CatBoost with DataFrame (mixed dtypes) — WORKS

In [6]:
X_df = pd.DataFrame({
    'num_col': [1.0, 2.0, 3.0, 4.0, 5.0],
    'cat_col': [0, 1, 0, 1, 0]  # int column
})
print(X_df.dtypes)

model = CatBoostRegressor(verbose=0, iterations=10)
try:
    model.fit(X_df, y, cat_features=[1])
    print('SUCCESS')
except Exception as e:
    print(f'ERROR: {e}')

num_col    float64
cat_col      int64
dtype: object
SUCCESS


## 4. CatBoost with DataFrame and string categories — WORKS

In [7]:
X_df_str = pd.DataFrame({
    'num_col': [1.0, 2.0, 3.0, 4.0, 5.0],
    'cat_col': ['a', 'b', 'a', 'b', 'a']
})
print(X_df_str.dtypes)

model = CatBoostRegressor(verbose=0, iterations=10)
try:
    model.fit(X_df_str, y, cat_features=[1])
    print('SUCCESS')
except Exception as e:
    print(f'ERROR: {e}')

num_col    float64
cat_col     object
dtype: object
SUCCESS


## 5. CatBoost with Pool — WORKS (explicit control)

In [8]:
from catboost import Pool

pool = Pool(X_float, y, cat_features=[1])
model = CatBoostRegressor(verbose=0, iterations=10)
try:
    model.fit(pool)
    print('SUCCESS')
except Exception as e:
    print(f'ERROR: {e}')

CatBoostError: 'data' is numpy array of floating point numerical type, it means no categorical features, but 'cat_features' parameter specifies nonzero number of categorical features

## 6. The ForecasterRecursive scenario

This reproduces what happens inside `ForecasterRecursive.fit()` with CatBoost and categorical exog.

In [ ]:
from skforecast.recursive import ForecasterRecursive

y_series = pd.Series(np.arange(20, dtype=float), name='y')
exog_df = pd.DataFrame({
    'exog_num': np.arange(100, 120, dtype=float),
    'exog_cat': pd.Categorical(range(20))
})

forecaster = ForecasterRecursive(
    estimator=CatBoostRegressor(verbose=0, iterations=10),
    lags=3,
    categorical_features='auto'
)

try:
    forecaster.fit(y=y_series, exog=exog_df)
    print('SUCCESS')
except Exception as e:
    print(f'ERROR: {e}')

## Summary

| Input type | cat_features | Result |
|---|---|---|
| numpy float64 | `[1]` | **ERROR** |
| numpy float64 (col cast to int but array still float) | `[1]` | **ERROR** |
| DataFrame (int cat column) | `[1]` | OK |
| DataFrame (str cat column) | `[1]` | OK |
| Pool (from float numpy) | `[1]` | OK |

**Root cause**: CatBoost checks `data.dtype.kind == 'f'` and if True, refuses `cat_features`. It needs either:
- A DataFrame where categorical columns have non-float dtype
- A Pool object
- A numpy array that is NOT float dtype

## Draft GitHub Issue for CatBoost

**Title:** `cat_features` rejected when input is a float numpy array — limits integration with ML frameworks

---

Hi CatBoost team,

I'm Javier, one of the developers of [skforecast](https://github.com/skforecast/skforecast), a Python library for time series forecasting with machine learning. We rely heavily on CatBoost as one of our recommended estimators, and we've recently run into a limitation that we'd like to discuss.

### Problem

When passing `cat_features` to `CatBoostRegressor.fit()` (or `CatBoostClassifier.fit()`), CatBoost raises an error if the input `X` is a **numpy float array**, even when the categorical columns contain valid integer-like values (e.g., `0.0`, `1.0`, `2.0` produced by `OrdinalEncoder`).

```python
import numpy as np
from catboost import CatBoostRegressor

X = np.array([
    [1.0, 0.0],
    [2.0, 1.0],
    [3.0, 0.0],
], dtype=np.float64)
y = np.array([10.0, 20.0, 15.0])

model = CatBoostRegressor(verbose=0, iterations=10)
model.fit(X, y, cat_features=[1])  # ERROR: cat_features with float array
```

### Why this matters for frameworks

Internally, skforecast (and many other ML frameworks) use **numpy arrays** for performance. A numpy array has a single dtype for the entire array, so if any column is float (which is always the case in regression problems), the entire array is `float64`. There is no way to have mixed dtypes in a numpy array.

This means that even though our categorical columns contain perfectly valid ordinal-encoded integer values (`0.0`, `1.0`, `2.0`...), CatBoost rejects them because it checks `data.dtype.kind == 'f'` and refuses `cat_features` for float arrays.

### Current workarounds (and why they're not ideal)

| Workaround | Issue |
|---|---|
| Pass a DataFrame instead of numpy | Significant performance penalty, especially with large datasets |
| Use a `Pool` object | Requires CatBoost-specific code paths, breaks estimator-agnostic design |
| Don't use `cat_features` at all | Loses the benefit of CatBoost's native categorical encoding |

### Suggestion

Would it be possible to relax the float-array restriction when `cat_features` is explicitly provided? If the user explicitly declares which columns are categorical, CatBoost could cast those specific columns to integers internally before applying its categorical encoding, rather than rejecting the entire input.

The values in those columns are guaranteed to be valid integer-like floats (e.g., `0.0`, `1.0`, `2.0`) since they come from standard preprocessing (e.g., `sklearn.preprocessing.OrdinalEncoder`).

This would make CatBoost much easier to integrate into frameworks that operate on numpy arrays internally, while preserving the benefit of CatBoost's native categorical handling.

Thank you for considering this. We love CatBoost and would like to fully support its native categorical features in skforecast.

Best regards,
Javier — skforecast team